# Plain-text NER

Compare small and large GLiNER on the 50 supplied documents. Only aggregate metrics are displayed.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.inference import DEFAULT_MODEL, LARGE_MODEL, load_gliner_model, predict_dataset
samples = load_jsonl(ROOT / 'data/plain/validation.jsonl')
print('Documents:', len(samples))

Documents: 50


In [2]:
for name, model_name in [('small', DEFAULT_MODEL), ('large', LARGE_MODEL)]:
    output = ROOT / f'results/plain/gliner_{name}.jsonl'
    if not output.exists():
        model = load_gliner_model(model_name)
        predictions = predict_dataset(model, samples, list(ENTITY_LABELS), threshold=0.5)
        save_responses_to_jsonl(predictions, output)
        del model
    print('Model:', model_name)
    strict = evaluate_ner_strict(ROOT / 'data/plain/validation.jsonl', output)
    relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation.jsonl', output)

Model: gliner-community/gliner_small-v2.5

STRICT EVALUATION
Micro -> P: 0.71 | R: 0.85 | F1: 0.77 | TP: 1740 | FP: 717 | FN: 310

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.95   1.00   0.97     500      28       0
ID_CARD_NUMBER              0.78   0.81   0.80     162      45      38
ADDRESS                     0.03   0.06   0.04      12     385     188
REGISTRATION_NUMBER         0.82   0.90   0.86     225      50      25
DATE                        0.97   0.99   0.98     398      12       2
AMOUNT                      0.71   0.85   0.77     171      71      29
ORGANIZATION                0.68   0.91   0.78     272     126      28

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.71 | R: 0.85 | F1: 0.77 | TP: 1740 | FP: 717 | FN: 310

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------

Strict scoring requires the exact start, end and label. Relaxed scoring compares labels and entity texts after whitespace normalization. Occurrences count repeated mentions; unique counts identical mentions once per document. Neither relaxed view ignores case, accents or OCR spelling errors.